<a href="https://colab.research.google.com/github/iabbey-commits/Financial-Analysis-Agent/blob/main/Fin_Analysis_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An Intelligent Agent For Financial Report Analysis.

Brief Description:

The financial analysis agent is an AI-powered system which is developed to simplify and automate the analysis of financial reports. It extracts key financial metrics from uploaded reports, organizes the data into searchable knowledge, and uses intelligent reasoning to provide insights, performance comparisons, trend analysis, and budget recommendations. By combining semantic retrieval, vector indexing, and AI-driven reasoning, the agent helps users quickly understand financial performance and make informed business decisions.

Architecture for the Agent.

In [ ]:
from logging import StrFormatStyle
from graphviz import Digraph

# Create directed graph
Agent5 = Digraph("Financial_AI_Agent", format="png")

Agent5.attr(rankdir='LR', size='6,5')
Agent5.attr('node', shape='box', style='rounded,filled', color='lightblue')

# Nodes
Agent5.node('A', 'Data Ingestion', shape='box', style='rounded,filled', color='red' )
Agent5.node('B', 'Table Segmentation')
Agent5.node('C', 'KPI Extraction', color='blue')
Agent5.node('D', 'Vector Indexing (FAISS)')
Agent5.node('E', 'Router Node', shape='circle', style='filled', color='yellow')
Agent5.node('F', 'Semantic Retrieval', shape='circle', style='filled', color='purple')
Agent5.node('G', 'Diagnostic Loop', shape='circle', style='filled', color='green')
Agent5.node('H', 'Reasoning Agent', shape='circle', style='rounded,filled', color='grey')
Agent5.node('I', 'Markdown Report Output', color='orange')

# Edges
Agent5.edge('A', 'B')
Agent5.edge('B', 'C')
Agent5.edge('C', 'D')
Agent5.edge('D', 'E')

# Conditional Branches
Agent5.edge('E', 'F', label='Simple Query')
Agent5.edge('E', 'G', label='Complex Analysis')

# Retrieval and reasoning
Agent5.edge('F', 'H')
Agent5.edge('G', 'F', label='Need More Data')
Agent5.edge('G', 'H', label='Sufficient Context')

# Final output
Agent5.edge('H', 'I')

# Export diagram
Agent5.render('financial_ai_agent_architecture', view=True)

In [ ]:
from IPython.display import Image
Image('financial_ai_agent_architecture.png')

Package Installation

In [ ]:
!pip install pandas numpy faiss-cpu sentence-transformers groq openpyxl python-dotenv

Uploading Files


In [ ]:
from google.colab import files

uploaded = files.upload()

Insert API Key


In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [ ]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

Loading Files


In [ ]:
import pandas as pd

file_name = list(uploaded.keys())[0]

df = pd.read_excel(file_name, header=None)

df.head(20)

Extracting Tables

In [ ]:
def extract_tables(df):
    tables = []
    current = []

    for _, row in df.iterrows():
        if row.isnull().all():
            if current:
                tables.append(pd.DataFrame(current))
                current = []
        else:
            current.append(row.tolist())

    if current:
        tables.append(pd.DataFrame(current))

    return tables

tables = extract_tables(df)

len(tables)

Understanding Tables by AI

In [ ]:
import json

def table_to_text(table):
    return table.fillna("").to_string(index=False)

def interpret_table(text):
    prompt = f"""
    Analyze this financial table:

    {text}

    Extract KPIs and return JSON only:

    {{
      "metrics": [
        {{
          "name": "",
          "actual": number or null,
          "budget": number or null,
          "prior": number or null
        }}
      ]
    }}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    result = response.choices[0].message.content

    try:
        # Attempt to extract JSON from a markdown code block first
        if '```json' in result and '```' in result:
            json_start = result.find('```json') + len('```json')
            json_end = result.find('```', json_start)
            if json_start != -1 and json_end != -1:
                json_str = result[json_start:json_end].strip()
                json.loads(json_str) # Validate JSON
                return json_str

        # Fallback to finding the first opening brace and last closing brace
        start = result.find('{')
        end = result.rfind('}')

        if start != -1 and end != -1 and start < end:
            json_str = result[start : end + 1]
            json.loads(json_str) # This will raise an error if not valid
            return json_str
        else:
            print(f"Warning: No valid JSON structure found in response: {result}")
            return json.dumps({"metrics": []}) # Return empty metrics
    except json.JSONDecodeError as e:
        print(f"JSONDecodeError extracting JSON from response: {e}, Raw response: {result}")
        return json.dumps({"metrics": []}) # Return empty metrics on parsing error
    except Exception as e:
        print(f"Other Error extracting JSON from response: {e}, Raw response: {result}")
        return json.dumps({"metrics": []}) # Return empty metrics on other errors


Processing All Tables


In [ ]:
import time

all_metrics = []

for table in tables:
    text = table_to_text(table)

    try:
        result = interpret_table(text)
        parsed = json.loads(result)

        if "metrics" in parsed:
            all_metrics.extend(parsed["metrics"])

    except Exception as e:
        print("Error:", e)

    # Add a delay to prevent rate limiting
    time.sleep(10) # Increased sleep time to 10 seconds for better rate limit handling

all_metrics[:10]

Building RAG System

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = []

# Coverting Metrix into Text Chunks.
for m in all_metrics:
    t = f"""
    Metric: {m.get('name')}
    Actual: {m.get('actual')}
    Budget: {m.get('budget')}
    Prior: {m.get('prior')}
    """
    texts.append(t)

texts[:3]

# Creating A Vector Index
embeddings = embed_model.encode(texts)

dim = embeddings.shape[1]

index = faiss.IndexFlatL2(dim)

index.add(np.array(embeddings))

# Define Retrival Index.
def retrieve(query, k=5):
    q_embed = embed_model.encode([query])

    distances, indices = index.search(q_embed, k)

    return [texts[i] for i in indices[0]]

Define Chat Function

In [ ]:
from IPython.display import Markdown

def ask_agent(question):
    context = retrieve(question)

    prompt = f"""
    You are a real estate asset management AI agent.

    Context:
    {context}

    Question:
    {question}

    Provide detailed financial insights. Format your response clearly using Markdown.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    # Return the Markdown directly for Colab to render
    return Markdown(response.choices[0].message.content)

Chart with Agent

In [ ]:
ask_agent("Extract KPIs")

In [ ]:
ask_agent("which properties are not performing?")

In [ ]:
ask_agent("Compare actual performance to budget")

In [ ]:
ask_agent("Compare actual performance to prior")

In [ ]:
ask_agent("Review the operation performance of the report")

In [ ]:
ask_agent("Create a new budget for the next year")

In [ ]:
ask_agent("Analyze the entire report")